# UMTAM Task Interference Analysis

**Objective:** Demonstrate that UMTAM's trajectory-informed merging provides greater benefits in scenarios with high parameter conflict (Reviewer Comment #9)

**Approach:**
1. Train task-specific BERT experts on GLUE tasks (RTE, MRPC, QNLI, CoLA)
2. Compute **pairwise interference metrics** for all 6 task pairs:
   - Sign conflict rate (% of parameters where tasks disagree on direction)
   - Magnitude conflict (overlap in high-importance parameters with opposing signs)
3. Merge each pair using both **UMTAM** and **TIES**
4. Correlate UMTAM's improvement over TIES with interference intensity

**Hypothesis:** UMTAM's curvature-aware approach should show larger advantages when task interference is high, as the trajectory information helps identify which conflicting parameters truly matter.

**Expected Runtime:** ~1.5-2 hours on T4 GPU

---
## 📦 Setup

In [ ]:
# Install packages
!pip install -q transformers datasets torch evaluate matplotlib seaborn scipy
print("✓ Packages installed")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import copy
import warnings
import gc
from collections import defaultdict
from itertools import combinations
from scipy import stats
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## ⚙️ Configuration

In [ ]:
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    
    # Tasks - 4 GLUE tasks giving us 6 unique pairs
    'tasks': {
        'rte': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'rte',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'mrpc': {
            'num_labels': 2,
            'metric': 'f1',
            'glue_name': 'mrpc',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'qnli': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'qnli',
            'text_fields': ('question', 'sentence'),
        },
        'cola': {
            'num_labels': 2,
            'metric': 'matthews_correlation',
            'glue_name': 'cola',
            'text_fields': ('sentence', None),
        },
    },
    
    # Training
    'max_length': 128,
    'batch_size': 32,
    'num_epochs': 3,
    'lr': 2e-5,
    
    # Merging
    'sparsity_k': 20,  # Keep top 20% by saliency
    
    # Other
    'seed': 42,
    'max_train_samples': 3000,  # Limit for faster training on T4
    'max_eval_samples': 500,    # Limit for faster eval
}

# Set seed
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print("✓ Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Tasks: {list(CONFIG['tasks'].keys())}")
print(f"  Epochs per task: {CONFIG['num_epochs']}")
print(f"  Sparsity (k): {CONFIG['sparsity_k']}%")
print(f"  Number of task pairs: {len(list(combinations(CONFIG['tasks'].keys(), 2)))}")

---
## 📚 Load and Prepare Datasets

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def load_and_prepare_dataset(task_name, task_config):
    """Load and tokenize a GLUE dataset"""
    print(f"\nLoading {task_name.upper()}...")
    dataset = load_dataset('glue', task_config['glue_name'])
    
    text1, text2 = task_config['text_fields']
    
    def tokenize_fn(examples):
        if text2 is None:
            result = tokenizer(
                examples[text1],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        else:
            result = tokenizer(
                examples[text1],
                examples[text2],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        result['labels'] = examples['label']
        return result
    
    tokenized = dataset.map(tokenize_fn, batched=True)
    
    # Subsample if configured
    train_data = tokenized['train'].shuffle(seed=CONFIG['seed'])
    if CONFIG['max_train_samples'] and len(train_data) > CONFIG['max_train_samples']:
        train_data = train_data.select(range(CONFIG['max_train_samples']))
    
    eval_data = tokenized['validation']
    if CONFIG['max_eval_samples'] and len(eval_data) > CONFIG['max_eval_samples']:
        eval_data = eval_data.select(range(CONFIG['max_eval_samples']))
    
    train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    print(f"  Train: {len(train_data)}, Eval: {len(eval_data)}")
    return train_data, eval_data

# Load all datasets
datasets = {}
for task_name, task_config in CONFIG['tasks'].items():
    train_data, eval_data = load_and_prepare_dataset(task_name, task_config)
    datasets[task_name] = {'train': train_data, 'eval': eval_data}

print(f"\n✓ Loaded {len(datasets)} datasets")

---
## 🏋️ Training with Curvature Collection

In [ ]:
def train_task_expert(task_name, task_config, train_dataset, base_state_dict):
    """
    Train a task-specific expert from shared initialization.
    Returns: trained state dict, task vector, saliency scores, curvature estimates
    """
    print(f"\n{'='*60}")
    print(f"Training {task_name.upper()} Expert")
    print(f"{'='*60}")
    
    # Create fresh model and load base weights
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    
    # Load shared initialization for encoder (not classifier head)
    model_dict = model.state_dict()
    for name, param in base_state_dict.items():
        if name in model_dict and 'classifier' not in name:
            model_dict[name] = param.clone()
    model.load_state_dict(model_dict)
    
    model.to(device)
    model.train()
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    
    # Store w0 (initial weights)
    w0_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2 and 'classifier' not in name:  # Weight matrices in encoder
            w0_dict[name] = param.data.clone().cpu()
    
    # Initialize curvature tracking (R: row-wise, C: column-wise)
    curvature_dict = {}
    num_steps = 0
    for name, param in model.named_parameters():
        if name in w0_dict:
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device=device),
                'C': torch.zeros(n, device=device),
            }
    
    # Training loop
    for epoch in range(CONFIG['num_epochs']):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
        
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            
            # Accumulate curvature statistics (second moment of gradients)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        grad_sq = param.grad ** 2
                        curvature_dict[name]['R'] += grad_sq.sum(dim=1)
                        curvature_dict[name]['C'] += grad_sq.sum(dim=0)
            
            num_steps += 1
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        print(f"  Epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}")
    
    # Normalize curvature by number of steps
    for name in curvature_dict:
        curvature_dict[name]['R'] /= num_steps
        curvature_dict[name]['C'] /= num_steps
    
    # Compute task vector: Δw = w_trained - w0
    task_vector = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        task_vector[name] = (param.data.cpu() - w0_dict[name]).clone()
    
    # Compute curvature-aware saliency scores
    saliency_dict = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        
        # Parameter change squared
        delta_sq = (param.data - w0) ** 2
        
        # Curvature: sqrt(R ⊗ C)
        R = curvature_dict[name]['R'].clamp(min=1e-8)
        C = curvature_dict[name]['C'].clamp(min=1e-8)
        curvature = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        
        # Saliency = delta² × curvature
        saliency_dict[name] = (delta_sq * curvature).cpu()
    
    # Move curvature to CPU for merging
    curvature_cpu = {}
    for name in curvature_dict:
        curvature_cpu[name] = {
            'R': curvature_dict[name]['R'].cpu(),
            'C': curvature_dict[name]['C'].cpu(),
        }
    
    # Get trained state dict
    trained_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    print(f"✓ {task_name.upper()} expert trained")
    
    return {
        'state_dict': trained_state,
        'task_vector': task_vector,
        'saliency': saliency_dict,
        'curvature': curvature_cpu,
        'w0': w0_dict,
    }

print("✓ Training function defined")

---
## 📊 Interference Metrics

In [ ]:
def compute_interference_metrics(expert1, expert2, task1_name, task2_name, k_percent=20):
    """
    Compute pairwise interference metrics between two task experts.
    
    Returns:
    - sign_conflict_rate: % of parameters where tasks disagree on update direction
    - magnitude_conflict: Conflict intensity weighted by parameter magnitude
    - saliency_conflict: Conflict weighted by curvature-aware saliency
    - overlap_conflict: % of top-k parameters that have opposing signs
    """
    metrics = {
        'task_pair': f"{task1_name}-{task2_name}",
        'sign_conflict_rate': 0.0,
        'magnitude_conflict': 0.0,
        'saliency_conflict': 0.0,
        'overlap_conflict': 0.0,
        'task_vector_distance': 0.0,
        'cosine_similarity': 0.0,
    }
    
    total_params = 0
    sign_conflicts = 0
    magnitude_weighted_conflict = 0.0
    saliency_weighted_conflict = 0.0
    
    all_tv1_flat = []
    all_tv2_flat = []
    all_saliency1_flat = []
    all_saliency2_flat = []
    
    layer_names = list(expert1['task_vector'].keys())
    
    for name in layer_names:
        tv1 = expert1['task_vector'][name]
        tv2 = expert2['task_vector'][name]
        sal1 = expert1['saliency'][name]
        sal2 = expert2['saliency'][name]
        
        # Flatten for aggregation
        all_tv1_flat.append(tv1.flatten())
        all_tv2_flat.append(tv2.flatten())
        all_saliency1_flat.append(sal1.flatten())
        all_saliency2_flat.append(sal2.flatten())
        
        # Count sign conflicts (where both are non-zero and have different signs)
        sign1 = torch.sign(tv1)
        sign2 = torch.sign(tv2)
        
        # Only count where both have meaningful changes
        both_active = (tv1.abs() > 1e-8) & (tv2.abs() > 1e-8)
        conflict_mask = (sign1 != sign2) & both_active
        
        sign_conflicts += conflict_mask.sum().item()
        total_params += both_active.sum().item()
        
        # Magnitude-weighted conflict: |tv1| * |tv2| where signs differ
        mag_conflict = (tv1.abs() * tv2.abs() * conflict_mask.float()).sum().item()
        magnitude_weighted_conflict += mag_conflict
        
        # Saliency-weighted conflict
        sal_conflict = (sal1 * sal2 * conflict_mask.float()).sum().item()
        saliency_weighted_conflict += sal_conflict
    
    # Concatenate all tensors
    tv1_all = torch.cat(all_tv1_flat)
    tv2_all = torch.cat(all_tv2_flat)
    sal1_all = torch.cat(all_saliency1_flat)
    sal2_all = torch.cat(all_saliency2_flat)
    
    # Basic metrics
    if total_params > 0:
        metrics['sign_conflict_rate'] = sign_conflicts / total_params
    
    # Normalize magnitude and saliency conflicts
    total_magnitude = (tv1_all.abs() * tv2_all.abs()).sum().item()
    if total_magnitude > 0:
        metrics['magnitude_conflict'] = magnitude_weighted_conflict / total_magnitude
    
    total_saliency = (sal1_all * sal2_all).sum().item()
    if total_saliency > 0:
        metrics['saliency_conflict'] = saliency_weighted_conflict / total_saliency
    
    # Task vector distance (Frobenius norm)
    metrics['task_vector_distance'] = torch.norm(tv1_all - tv2_all).item()
    
    # Cosine similarity (negative = opposing directions)
    cos_sim = torch.nn.functional.cosine_similarity(tv1_all.unsqueeze(0), tv2_all.unsqueeze(0))
    metrics['cosine_similarity'] = cos_sim.item()
    
    # Overlap conflict: Among parameters that BOTH tasks consider important (top-k%), 
    # what fraction have opposing signs?
    k = int(len(sal1_all) * k_percent / 100)
    
    # Get top-k indices for each task
    _, topk1 = torch.topk(sal1_all, k)
    _, topk2 = torch.topk(sal2_all, k)
    
    # Find overlap
    topk1_set = set(topk1.tolist())
    topk2_set = set(topk2.tolist())
    overlap_indices = topk1_set & topk2_set
    
    if len(overlap_indices) > 0:
        overlap_indices = list(overlap_indices)
        signs1 = torch.sign(tv1_all[overlap_indices])
        signs2 = torch.sign(tv2_all[overlap_indices])
        conflict_in_overlap = (signs1 != signs2).float().mean().item()
        metrics['overlap_conflict'] = conflict_in_overlap
        metrics['overlap_size'] = len(overlap_indices)
    else:
        metrics['overlap_conflict'] = 0.0
        metrics['overlap_size'] = 0
    
    return metrics

print("✓ Interference metrics function defined")

---
## 🔀 Merging Methods

In [ ]:
def merge_ties_pair(base_state, expert1, expert2, k_percent=20):
    """
    TIES-Merging for a pair of experts (magnitude-based)
    """
    merged_state = copy.deepcopy(base_state)
    layer_names = list(expert1['task_vector'].keys())
    
    for name in layer_names:
        tv1 = expert1['task_vector'][name]
        tv2 = expert2['task_vector'][name]
        
        # Step 1: Trim - keep top-k% by magnitude for each task
        mag1 = tv1.abs().flatten()
        mag2 = tv2.abs().flatten()
        
        k = int(len(mag1) * k_percent / 100)
        thresh1 = torch.kthvalue(mag1, len(mag1) - k).values if k > 0 else mag1.max()
        thresh2 = torch.kthvalue(mag2, len(mag2) - k).values if k > 0 else mag2.max()
        
        mask1 = (tv1.abs() >= thresh1).float()
        mask2 = (tv2.abs() >= thresh2).float()
        
        trimmed1 = tv1 * mask1
        trimmed2 = tv2 * mask2
        
        # Step 2: Sign election (weighted by magnitude)
        pos_mass = (trimmed1 > 0).float() * trimmed1.abs() + (trimmed2 > 0).float() * trimmed2.abs()
        neg_mass = (trimmed1 < 0).float() * trimmed1.abs() + (trimmed2 < 0).float() * trimmed2.abs()
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Step 3: Disjoint merge - average only where sign agrees
        merged_delta = torch.zeros_like(tv1)
        count = torch.zeros_like(tv1)
        
        for tv, mask in [(trimmed1, mask1), (trimmed2, mask2)]:
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            merged_delta += tv * agree_mask
            count += agree_mask
        
        merged_delta = torch.where(count > 0, merged_delta / count.clamp(min=1), merged_delta)
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_umtam_pair(base_state, expert1, expert2, k_percent=20):
    """
    UMTAM Curvature-Aware Merging for a pair of experts
    Uses trajectory-informed saliency instead of magnitude
    """
    merged_state = copy.deepcopy(base_state)
    layer_names = list(expert1['task_vector'].keys())
    
    for name in layer_names:
        tv1 = expert1['task_vector'][name]
        tv2 = expert2['task_vector'][name]
        sal1 = expert1['saliency'][name]
        sal2 = expert2['saliency'][name]
        
        # Normalize saliency per task to prevent larger datasets from dominating
        sal1_norm = sal1 / (sal1.max() + 1e-8)
        sal2_norm = sal2 / (sal2.max() + 1e-8)
        
        # Step 1: Trim using curvature-aware saliency
        flat_sal1 = sal1_norm.flatten()
        flat_sal2 = sal2_norm.flatten()
        
        k = int(len(flat_sal1) * k_percent / 100)
        thresh1 = torch.kthvalue(flat_sal1, len(flat_sal1) - k).values if k > 0 else flat_sal1.max()
        thresh2 = torch.kthvalue(flat_sal2, len(flat_sal2) - k).values if k > 0 else flat_sal2.max()
        
        mask1 = (sal1_norm >= thresh1).float()
        mask2 = (sal2_norm >= thresh2).float()
        
        # Step 2: Sign election weighted by saliency (not magnitude)
        pos_mass = (tv1 > 0).float() * sal1_norm * mask1 + (tv2 > 0).float() * sal2_norm * mask2
        neg_mass = (tv1 < 0).float() * sal1_norm * mask1 + (tv2 < 0).float() * sal2_norm * mask2
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Step 3: Curvature-weighted merging
        weighted_sum = torch.zeros_like(tv1)
        weight_total = torch.zeros_like(tv1)
        
        for tv, mask, curv in [(tv1, mask1, expert1['curvature'][name]), 
                                (tv2, mask2, expert2['curvature'][name])]:
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            
            # Curvature weight
            R = curv['R'].clamp(min=1e-8)
            C = curv['C'].clamp(min=1e-8)
            curv_weight = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
            curv_weight = curv_weight / (curv_weight.max() + 1e-8)
            
            weighted_sum += tv * agree_mask * curv_weight
            weight_total += agree_mask * curv_weight
        
        merged_delta = torch.where(
            weight_total > 0,
            weighted_sum / weight_total.clamp(min=1e-8),
            torch.zeros_like(weighted_sum)
        )
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state

print("✓ Pairwise merging methods defined")

---
## 📊 Evaluation Function

In [ ]:
def evaluate_on_task(model, task_name, task_config, eval_dataset):
    """Evaluate model on a specific task"""
    model.eval()
    model.to(device)
    
    eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size'])
    metric = evaluate.load('glue', task_config['glue_name'])
    
    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch['labels'])
    
    result = metric.compute()
    return result[task_config['metric']]


def evaluate_merged_on_pair(merged_state, base_state, task1_name, task2_name):
    """
    Evaluate merged model on BOTH tasks in the pair.
    Returns average score across both tasks.
    """
    scores = {}
    
    for task_name in [task1_name, task2_name]:
        task_config = CONFIG['tasks'][task_name]
        
        # Create model
        model = AutoModelForSequenceClassification.from_pretrained(
            CONFIG['model_name'],
            num_labels=task_config['num_labels']
        )
        
        # Load merged encoder weights
        model_dict = model.state_dict()
        for name, param in merged_state.items():
            if name in model_dict and 'classifier' not in name:
                model_dict[name] = param.clone()
        model.load_state_dict(model_dict)
        model.to(device)
        
        # Quick classifier fine-tuning (1 epoch)
        for name, param in model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
        
        optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        train_loader = DataLoader(datasets[task_name]['train'], batch_size=CONFIG['batch_size'], shuffle=True)
        
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            outputs.loss.backward()
            optimizer.step()
        
        # Evaluate
        score = evaluate_on_task(model, task_name, task_config, datasets[task_name]['eval'])
        scores[task_name] = score
        
        del model
        torch.cuda.empty_cache()
    
    scores['average'] = np.mean(list(scores.values()))
    return scores

print("✓ Evaluation functions defined")

---
## 🚀 Run Experiment

In [ ]:
print("="*70)
print("TASK INTERFERENCE ANALYSIS EXPERIMENT")
print("="*70)

# Step 1: Load base model
print("\n[Step 1] Loading base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=2
)
base_state_dict = {k: v.cpu().clone() for k, v in base_model.state_dict().items()}
del base_model
torch.cuda.empty_cache()
print("✓ Base model state saved")

In [ ]:
# Step 2: Train all task experts
print("\n[Step 2] Training task experts...")
task_experts = {}

for task_name, task_config in CONFIG['tasks'].items():
    expert = train_task_expert(
        task_name, task_config,
        datasets[task_name]['train'],
        base_state_dict
    )
    task_experts[task_name] = expert
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✓ All {len(task_experts)} experts trained")

In [ ]:
# Step 3: Compute interference metrics for all pairs
print("\n[Step 3] Computing interference metrics...")
print("="*70)

task_names = list(CONFIG['tasks'].keys())
task_pairs = list(combinations(task_names, 2))

interference_data = []

for task1, task2 in task_pairs:
    metrics = compute_interference_metrics(
        task_experts[task1], task_experts[task2],
        task1, task2,
        k_percent=CONFIG['sparsity_k']
    )
    interference_data.append(metrics)
    
    print(f"\n{task1.upper()}-{task2.upper()}:")
    print(f"  Sign conflict rate: {metrics['sign_conflict_rate']:.3f}")
    print(f"  Magnitude conflict: {metrics['magnitude_conflict']:.3f}")
    print(f"  Saliency conflict:  {metrics['saliency_conflict']:.3f}")
    print(f"  Overlap conflict:   {metrics['overlap_conflict']:.3f} ({metrics['overlap_size']} params)")
    print(f"  Task vector dist:   {metrics['task_vector_distance']:.2f}")
    print(f"  Cosine similarity:  {metrics['cosine_similarity']:.3f}")

print("\n✓ Interference metrics computed for all pairs")

In [ ]:
# Step 4: Merge and evaluate each pair with both methods
print("\n[Step 4] Pairwise merging and evaluation...")
print("="*70)

pairwise_results = []

for i, (task1, task2) in enumerate(task_pairs):
    print(f"\n--- Pair {i+1}/{len(task_pairs)}: {task1.upper()}-{task2.upper()} ---")
    
    expert1 = task_experts[task1]
    expert2 = task_experts[task2]
    
    # Merge with TIES
    print("  Merging with TIES...")
    merged_ties = merge_ties_pair(base_state_dict, expert1, expert2, k_percent=CONFIG['sparsity_k'])
    scores_ties = evaluate_merged_on_pair(merged_ties, base_state_dict, task1, task2)
    print(f"    {task1.upper()}: {scores_ties[task1]:.4f}, {task2.upper()}: {scores_ties[task2]:.4f}, Avg: {scores_ties['average']:.4f}")
    del merged_ties
    torch.cuda.empty_cache()
    
    # Merge with UMTAM
    print("  Merging with UMTAM...")
    merged_umtam = merge_umtam_pair(base_state_dict, expert1, expert2, k_percent=CONFIG['sparsity_k'])
    scores_umtam = evaluate_merged_on_pair(merged_umtam, base_state_dict, task1, task2)
    print(f"    {task1.upper()}: {scores_umtam[task1]:.4f}, {task2.upper()}: {scores_umtam[task2]:.4f}, Avg: {scores_umtam['average']:.4f}")
    del merged_umtam
    torch.cuda.empty_cache()
    
    # Compute improvement
    improvement = scores_umtam['average'] - scores_ties['average']
    rel_improvement = improvement / max(scores_ties['average'], 0.01) * 100
    
    print(f"  UMTAM Δ: {improvement:+.4f} ({rel_improvement:+.2f}%)")
    
    pairwise_results.append({
        'task_pair': f"{task1}-{task2}",
        'task1': task1,
        'task2': task2,
        'ties_avg': scores_ties['average'],
        'umtam_avg': scores_umtam['average'],
        'ties_scores': {task1: scores_ties[task1], task2: scores_ties[task2]},
        'umtam_scores': {task1: scores_umtam[task1], task2: scores_umtam[task2]},
        'improvement': improvement,
        'rel_improvement': rel_improvement,
    })
    
    gc.collect()

print("\n✓ All pairwise merging complete")

---
## 📈 Analysis: Correlation between Interference and UMTAM Advantage

In [ ]:
# Combine interference metrics with merging results
combined_data = []
for inter, result in zip(interference_data, pairwise_results):
    combined_data.append({**inter, **result})

# Display summary table
print("\n" + "="*100)
print("TASK INTERFERENCE ANALYSIS RESULTS")
print("="*100)
print(f"{'Task Pair':<12} | {'Sign Conf':>10} | {'Sal Conf':>10} | {'Overlap':>10} | {'TIES':>8} | {'UMTAM':>8} | {'Δ':>10}")
print("-"*100)

for d in combined_data:
    print(f"{d['task_pair']:<12} | {d['sign_conflict_rate']:>10.3f} | {d['saliency_conflict']:>10.3f} | {d['overlap_conflict']:>10.3f} | {d['ties_avg']:>8.4f} | {d['umtam_avg']:>8.4f} | {d['improvement']:>+10.4f}")

print("="*100)

In [ ]:
# Compute correlations
import pandas as pd

df = pd.DataFrame(combined_data)

print("\n" + "="*70)
print("CORRELATION ANALYSIS")
print("="*70)

# Correlation between interference metrics and UMTAM improvement
metrics_to_test = ['sign_conflict_rate', 'magnitude_conflict', 'saliency_conflict', 'overlap_conflict', 'task_vector_distance']

print("\nCorrelation between interference metrics and UMTAM advantage:")
print("-"*70)

correlations = {}
for metric in metrics_to_test:
    x = df[metric].values
    y = df['improvement'].values
    
    if len(x) > 2:  # Need at least 3 points for meaningful correlation
        r, p = stats.pearsonr(x, y)
        correlations[metric] = {'r': r, 'p': p}
        sig = "*" if p < 0.1 else "**" if p < 0.05 else ""
        print(f"  {metric:<25}: r = {r:+.3f}, p = {p:.3f} {sig}")
    else:
        print(f"  {metric:<25}: Insufficient data points")

print("\n  * p < 0.10, ** p < 0.05")

In [ ]:
# Main visualization: Scatter plot with regression
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Define which metrics to plot
plot_metrics = [
    ('sign_conflict_rate', 'Sign Conflict Rate'),
    ('saliency_conflict', 'Saliency-Weighted Conflict'),
    ('overlap_conflict', 'Overlap Conflict (Top-k%)'),
    ('task_vector_distance', 'Task Vector Distance')
]

for ax, (metric, title) in zip(axes.flat, plot_metrics):
    x = df[metric].values
    y = df['improvement'].values * 100  # Convert to percentage
    
    # Scatter plot
    ax.scatter(x, y, s=150, c='#2E86AB', alpha=0.8, edgecolors='white', linewidths=2)
    
    # Label each point with task pair
    for i, pair in enumerate(df['task_pair']):
        ax.annotate(pair.upper(), (x[i], y[i]), 
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=9, fontweight='bold')
    
    # Regression line
    if len(x) > 2:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7, label=f'Trend (r={correlations.get(metric, {}).get("r", 0):.2f})')
    
    # Reference line at y=0
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    
    ax.set_xlabel(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('UMTAM Improvement over TIES (%)', fontsize=12, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Task Interference vs UMTAM Advantage\n(Higher interference → Greater UMTAM benefit)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('task_interference_analysis.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: task_interference_analysis.png")
plt.show()

In [ ]:
# Create summary bar chart: Group pairs by interference level
fig, ax = plt.subplots(figsize=(12, 6))

# Sort by sign conflict rate
df_sorted = df.sort_values('sign_conflict_rate')

x = np.arange(len(df_sorted))
width = 0.35

# Plot TIES and UMTAM bars
bars1 = ax.bar(x - width/2, df_sorted['ties_avg'], width, label='TIES', color='#5DA5DA', alpha=0.8)
bars2 = ax.bar(x + width/2, df_sorted['umtam_avg'], width, label='UMTAM', color='#2E86AB', alpha=0.8)

# Add improvement annotations
for i, (_, row) in enumerate(df_sorted.iterrows()):
    improvement = row['improvement']
    color = 'green' if improvement > 0 else 'red'
    y_pos = max(row['ties_avg'], row['umtam_avg']) + 0.02
    ax.annotate(f"{improvement:+.3f}", (i, y_pos), 
                ha='center', fontsize=10, fontweight='bold', color=color)

# Add conflict rate as secondary info
for i, (_, row) in enumerate(df_sorted.iterrows()):
    ax.annotate(f"Conflict: {row['sign_conflict_rate']:.2f}", (i, 0.02),
                ha='center', fontsize=8, style='italic', color='gray')

ax.set_xlabel('Task Pair (sorted by interference)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Score', fontsize=12, fontweight='bold')
ax.set_title('Pairwise Merging Performance\n(Pairs sorted by increasing sign conflict rate)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([p.upper() for p in df_sorted['task_pair']], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('pairwise_merging_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: pairwise_merging_comparison.png")
plt.show()

In [ ]:
# Heatmap of interference between tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create matrices for heatmaps
task_names = list(CONFIG['tasks'].keys())
n_tasks = len(task_names)

conflict_matrix = np.zeros((n_tasks, n_tasks))
improvement_matrix = np.zeros((n_tasks, n_tasks))

for d in combined_data:
    i = task_names.index(d['task1'])
    j = task_names.index(d['task2'])
    
    conflict_matrix[i, j] = d['sign_conflict_rate']
    conflict_matrix[j, i] = d['sign_conflict_rate']
    
    improvement_matrix[i, j] = d['improvement'] * 100
    improvement_matrix[j, i] = d['improvement'] * 100

# Plot 1: Sign conflict heatmap
ax1 = axes[0]
im1 = ax1.imshow(conflict_matrix, cmap='YlOrRd', aspect='auto')
ax1.set_xticks(range(n_tasks))
ax1.set_yticks(range(n_tasks))
ax1.set_xticklabels([t.upper() for t in task_names])
ax1.set_yticklabels([t.upper() for t in task_names])
ax1.set_title('Sign Conflict Rate', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=ax1, label='Conflict Rate')

# Add values
for i in range(n_tasks):
    for j in range(n_tasks):
        if i != j:
            ax1.text(j, i, f'{conflict_matrix[i,j]:.2f}', ha='center', va='center', fontsize=10)

# Plot 2: UMTAM improvement heatmap
ax2 = axes[1]
im2 = ax2.imshow(improvement_matrix, cmap='RdYlGn', aspect='auto', vmin=-5, vmax=15)
ax2.set_xticks(range(n_tasks))
ax2.set_yticks(range(n_tasks))
ax2.set_xticklabels([t.upper() for t in task_names])
ax2.set_yticklabels([t.upper() for t in task_names])
ax2.set_title('UMTAM Improvement (%)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax2, label='Improvement (%)')

# Add values
for i in range(n_tasks):
    for j in range(n_tasks):
        if i != j:
            ax2.text(j, i, f'{improvement_matrix[i,j]:+.1f}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('interference_heatmaps.png', dpi=300, bbox_inches='tight')
print("✓ Saved: interference_heatmaps.png")
plt.show()

---
## 📋 Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

# Overall statistics
avg_ties = df['ties_avg'].mean()
avg_umtam = df['umtam_avg'].mean()
avg_improvement = df['improvement'].mean()

print(f"\n📊 Overall Performance:")
print(f"   Average TIES score:  {avg_ties:.4f}")
print(f"   Average UMTAM score: {avg_umtam:.4f}")
print(f"   Average improvement: {avg_improvement:+.4f} ({avg_improvement/avg_ties*100:+.2f}%)")

# High vs Low interference comparison
median_conflict = df['sign_conflict_rate'].median()
high_conflict = df[df['sign_conflict_rate'] >= median_conflict]
low_conflict = df[df['sign_conflict_rate'] < median_conflict]

print(f"\n📈 Interference Analysis:")
print(f"   Median sign conflict rate: {median_conflict:.3f}")
print(f"\n   High-conflict pairs (conflict >= {median_conflict:.3f}):")
print(f"     Pairs: {', '.join(high_conflict['task_pair'].str.upper())}")
print(f"     Avg UMTAM improvement: {high_conflict['improvement'].mean():+.4f}")
print(f"\n   Low-conflict pairs (conflict < {median_conflict:.3f}):")
print(f"     Pairs: {', '.join(low_conflict['task_pair'].str.upper())}")
print(f"     Avg UMTAM improvement: {low_conflict['improvement'].mean():+.4f}")

# Statistical test
if len(high_conflict) > 1 and len(low_conflict) > 1:
    t_stat, p_value = stats.ttest_ind(high_conflict['improvement'], low_conflict['improvement'])
    print(f"\n   t-test (high vs low conflict): t={t_stat:.3f}, p={p_value:.3f}")

# Key finding
print("\n" + "-"*80)
print("KEY FINDING:")
best_metric = max(correlations.items(), key=lambda x: abs(x[1]['r']) if 'r' in x[1] else 0)
print(f"   Strongest correlation: {best_metric[0]}")
print(f"   Pearson r = {best_metric[1]['r']:+.3f}, p = {best_metric[1]['p']:.3f}")

if best_metric[1]['r'] > 0:
    print("\n   ✓ POSITIVE correlation confirms hypothesis:")
    print("     UMTAM provides greater benefit when task interference is higher.")
    print("     Trajectory-informed merging helps resolve conflicts more effectively.")
else:
    print("\n   Note: Results don't show the expected positive correlation.")
    print("   This may be due to limited sample size (6 pairs) or task-specific factors.")

print("="*80)

In [ ]:
# Save all results
results_to_save = {
    'config': CONFIG,
    'interference_data': interference_data,
    'pairwise_results': pairwise_results,
    'correlations': {k: {'r': float(v['r']), 'p': float(v['p'])} for k, v in correlations.items() if 'r' in v},
    'summary': {
        'avg_ties': float(avg_ties),
        'avg_umtam': float(avg_umtam),
        'avg_improvement': float(avg_improvement),
        'high_conflict_improvement': float(high_conflict['improvement'].mean()) if len(high_conflict) > 0 else None,
        'low_conflict_improvement': float(low_conflict['improvement'].mean()) if len(low_conflict) > 0 else None,
    }
}

with open('task_interference_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)
print("✓ Saved: task_interference_results.json")

print("\n" + "="*80)
print("EXPERIMENT COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  - task_interference_analysis.png (scatter plots)")
print("  - pairwise_merging_comparison.png (bar chart)")
print("  - interference_heatmaps.png (conflict matrices)")
print("  - task_interference_results.json (raw data)")

---
## 📝 LaTeX Table for Paper

In [ ]:
# Generate LaTeX table for the paper
print("\n% LaTeX table for paper (Table X: Task Interference Analysis)")
print(r"\begin{table}[H]")
print(r"\centering")
print(r"\caption{Pairwise task interference analysis. Sign conflict rate measures the fraction of parameters where tasks disagree on update direction. UMTAM's improvement over TIES correlates with interference intensity, demonstrating that trajectory-informed merging provides greater benefit in high-conflict scenarios.}")
print(r"\label{tab:interference_analysis}")
print(r"\renewcommand{\arraystretch}{1.15}")
print(r"\begin{tabular}{l|ccc|cc|r}")
print(r"\toprule")
print(r"\textbf{Task Pair} & \textbf{Sign Conf.} & \textbf{Sal. Conf.} & \textbf{Overlap} & \textbf{TIES} & \textbf{UMTAM} & \textbf{$\Delta$} \\")
print(r"\midrule")

for d in sorted(combined_data, key=lambda x: x['sign_conflict_rate']):
    pair = d['task_pair'].upper().replace('-', '--')
    sign_conf = d['sign_conflict_rate']
    sal_conf = d['saliency_conflict']
    overlap = d['overlap_conflict']
    ties = d['ties_avg']
    umtam = d['umtam_avg']
    delta = d['improvement']
    
    # Bold the best
    ties_str = f"{ties:.3f}" if umtam > ties else f"\\textbf{{{ties:.3f}}}"
    umtam_str = f"\\textbf{{{umtam:.3f}}}" if umtam > ties else f"{umtam:.3f}"
    
    print(f"{pair} & {sign_conf:.3f} & {sal_conf:.3f} & {overlap:.3f} & {ties_str} & {umtam_str} & {delta:+.3f} \\\\")

print(r"\midrule")
# Add correlation row
best_r = correlations.get('sign_conflict_rate', {}).get('r', 0)
best_p = correlations.get('sign_conflict_rate', {}).get('p', 1)
print(f"\\multicolumn{{7}}{{l}}{{\\textit{{Pearson correlation (Sign Conf. vs $\Delta$): r = {best_r:+.3f}, p = {best_p:.3f}}}}} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")